In [ ]:
!pip install kaggle tqdm matplotlib

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json

import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Download dataset
!kaggle datasets download -d phylake1337/fire-dataset -p ./data --quiet

# Extract
import zipfile
with zipfile.ZipFile("./data/fire-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("./data")

print("Dataset Ready!")

In [ ]:
class FireDataset(Dataset):
    def __init__(self, img_dir, mask_dir, image_size=(256,256)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.image_size = image_size

        self.images = sorted(os.listdir(img_dir))

        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5,)*3, (0.5,)*3)
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        orig_w, orig_h = image.size

        image = self.transform(image)
        mask = mask.resize(self.image_size)

        mask = np.array(mask)
        mask = (mask > 0).astype(np.uint8)  # binary
        mask = torch.tensor(mask, dtype=torch.long)

        return image, mask

In [ ]:
train_dataset = FireDataset(
    img_dir="./data/images",
    mask_dir="./data/masks"
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [ ]:
class VGG16_Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True).features

        self.stage1 = vgg[:5]
        self.stage2 = vgg[5:10]
        self.stage3 = vgg[10:17]
        self.stage4 = vgg[17:24]
        self.stage5 = vgg[24:31]

    def forward(self, x):
        f1 = self.stage1(x)
        f2 = self.stage2(f1)
        f3 = self.stage3(f2)
        f4 = self.stage4(f3)
        f5 = self.stage5(f4)

        return [f1, f2, f3, f4, f5]

In [ ]:
class Hypercolumn(nn.Module):
    def forward(self, features):
        target_size = features[0].shape[2:]

        upsampled = [
            F.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
            for f in features
        ]

        return torch.cat(upsampled, dim=1)

In [ ]:
def sample_pixels(hc, mask, num_samples=2048):
    B, C, H, W = hc.shape

    hc = hc.permute(0,2,3,1).reshape(-1, C)
    mask = mask.view(-1)

    idx = torch.randperm(hc.shape[0])[:num_samples]

    return hc[idx], mask[idx]

In [ ]:
class PixelClassifier(nn.Module):
    def __init__(self, in_channels, num_classes=2):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)

In [ ]:
backbone = VGG16_Backbone().to(device)
hyper = Hypercolumn().to(device)

classifier = PixelClassifier(1472, 2).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(backbone.parameters()) + list(classifier.parameters()),
    lr=1e-4
)

for epoch in range(5):
    backbone.train()
    classifier.train()

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for images, masks in loop:
        images, masks = images.to(device), masks.to(device)

        features = backbone(images)
        hc = hyper(features)

        feats, labels = sample_pixels(hc, masks)

        outputs = classifier(feats)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

In [ ]:
def visualize(image, mask, pred):
    image = image.permute(1,2,0).cpu().numpy()
    image = (image * 0.5) + 0.5

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(image)
    plt.title("Image")

    plt.subplot(1,3,2)
    plt.imshow(mask, cmap='gray')
    plt.title("Ground Truth")

    plt.subplot(1,3,3)
    plt.imshow(pred, cmap='hot')
    plt.title("Prediction")

    plt.show()

In [ ]:
def visualize(image, mask, pred):
    image = image.permute(1,2,0).cpu().numpy()
    image = (image * 0.5) + 0.5

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(image)
    plt.title("Image")

    plt.subplot(1,3,2)
    plt.imshow(mask, cmap='gray')
    plt.title("Ground Truth")

    plt.subplot(1,3,3)
    plt.imshow(pred, cmap='hot')
    plt.title("Prediction")

    plt.show()